In [1]:
!pip install -q kaggle

In [2]:
import os
import zipfile

In [3]:
# Download the dataset from kaggle

os.environ["KAGGLE_USERNAME"] = "dorcaskagwiria"
os.environ["KAGGLE_KEY"] = "KGAT_148c83ee7198284e80213f155a253a7c"

!kaggle datasets download -d yashpwrr/resume-ner-training-dataset

with zipfile.ZipFile('resume-ner-training-dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('resume_ner_training_dataset')

print("Dataset downloaded and unzipped")

Dataset URL: https://www.kaggle.com/datasets/yashpwrr/resume-ner-training-dataset
License(s): MIT
100% 14.2M/14.2M [00:01<00:00, 11.5MB/s]

Dataset downloaded and unzipped


In [4]:
import json
import re
import random
import spacy
from spacy.tokens import DocBin

In [5]:
# Open the file in read mode
with open('resume_ner_training_dataset/train.json', 'r') as file:
    dataset = json.load(file)

In [6]:
# 1. Load a blank English spaCy model for tokenization
nlp = spacy.blank("en")

In [7]:
def clean_text(text): #data cleaning
    """
    Cleans structural noise and URLs while preserving exact
    character counts for index alignment.
    """
    # urls = re.findall(r'https?://\S+|www\.\S+', text)
    # for url in urls:
    #     text = text.replace(url, " " * len(url))
    text = re.sub(r'[\t\r\n]', ' ', text)
    text = re.sub(r'[●•❖➣]', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)

    return text

In [8]:
def convert_to_docbin(dataset):
    """
    Cleans text, filters entities, aligns indices to tokens,
    and packages data into a spaCy DocBin container.
    """
    db = DocBin()

    for record in dataset:
        text = record.get("text", "")
        annotations = record.get("annotations", [])

        cleaned_text = clean_text(text)
        doc = nlp.make_doc(cleaned_text)

        # IMPORTANT: initialize this for every document
        valid_entities = []

        # Sort annotations by start position
        sorted_annotations = sorted(annotations, key=lambda x: x[0])
        last_end_offset = -1

        for start, end, label in sorted_annotations:
            # 1. Validate raw boundaries
            if start < 0 or end > len(cleaned_text) or start >= end:
                continue

            # 2. Extract and strip whitespace safely
            entity_text = cleaned_text[start:end]
            l_offset = len(entity_text) - len(entity_text.lstrip())
            r_offset = len(entity_text) - len(entity_text.rstrip())

            start += l_offset
            end -= r_offset

            # Re-check bounds after stripping
            if start >= end:
                continue

            # 3. Drop overlapping entities
            if start < last_end_offset:
                continue

            # 4. Align to token boundaries
            # Using alignment_mode="contract" ensures no leading/trailing spaces in spans
            span = doc.char_span(
                start,
                end,
                label=label.upper().strip(),
                alignment_mode="contract"
            )

            # Skip if the span could not be aligned to tokens
            if span is None:
                continue

            # 5. Trim leading whitespace/punct tokens
            start_token_idx = 0
            while start_token_idx < len(span) and (span[start_token_idx].is_space or not span[start_token_idx].text.strip()):
                start_token_idx += 1

            # 6. Trim trailing whitespace/punct tokens
            end_token_idx = len(span)
            while end_token_idx > start_token_idx and (span[end_token_idx - 1].is_space or not span[end_token_idx - 1].text.strip()):
                end_token_idx -= 1

            if start_token_idx >= end_token_idx:
                continue

            # Slice the clean sub-span from the document
            # Returns the cleaned span
            span = doc[span.start + start_token_idx : span.start + end_token_idx]

            # 5. Final check: ensure span is valid and has non-whitespace bounds
            if span is not None and len(span.text.strip()) > 0:
                valid_entities.append(span)
                last_end_offset = span.end_char

        # Assign entities
        doc.ents = valid_entities

        # Add document to DocBin
        db.add(doc)

    return db

In [9]:
# --------------------------------------------------
# 1. Shuffle the dataset
# --------------------------------------------------

random.seed(42)
random.shuffle(dataset)


# --------------------------------------------------
# 2. Calculate split sizes
# --------------------------------------------------

total = len(dataset)

train_size = int(0.80 * total)
val_size = int(0.10 * total)

train_data = dataset[:train_size]
val_data = dataset[train_size:train_size + val_size]
test_data = dataset[train_size + val_size:]


# --------------------------------------------------
# 3. Convert each split into a DocBin
# --------------------------------------------------

train_db = convert_to_docbin(train_data)
val_db = convert_to_docbin(val_data)
test_db = convert_to_docbin(test_data)


# --------------------------------------------------
# 4. Save the DocBins
# --------------------------------------------------

train_db.to_disk("train.spacy")
val_db.to_disk("validation.spacy")
test_db.to_disk("test.spacy")


# --------------------------------------------------
# 5. Check the sizes
# --------------------------------------------------

print(f"Total records:      {total}")
print(f"Training records:   {len(train_data)}")
print(f"Validation records: {len(val_data)}")
print(f"Test records:       {len(test_data)}")

Total records:      5960
Training records:   4768
Validation records: 596
Test records:       596


In [10]:
!python -m spacy init config config.cfg --lang en --pipeline ner --optimize efficiency

⚠ To generate a more effective transformer-based config (GPU-only),
install the spacy-transformers package and re-run this command. The config
generated now does not use transformers.
ℹ Generated config template specific for your use case
- Language: en
- Pipeline: ner
- Optimize for: efficiency
- Hardware: CPU
- Transformer: None
✔ Auto-filled config with all values
✔ Saved config
config.cfg
You can now add your data and train your pipeline:
python -m spacy train config.cfg --paths.train ./train.spacy --paths.dev ./dev.spacy


In [11]:
!python -m spacy debug data config.cfg --paths.train ./train.spacy --paths.dev ./validation.spacy


============================ Data file validation ============================
✔ Pipeline can be initialized with data
✔ Corpus is loadable

=============================== Training stats ===============================
Language: en
Training pipeline: tok2vec, ner
4768 training docs
596 evaluation docs
⚠ 236 training examples also in evaluation data

============================== Vocab & Vectors ==============================
ℹ 3553528 total word(s) in the data (117313 unique)
ℹ No word vectors present in the package

========================== Named Entity Recognition ==========================
ℹ 0 label(s)
0 missing value(s) (tokens with '-' label)
✔ Good amount of examples for all labels
✔ Examples without occurrences available for all labels
✔ No entities consisting of or starting/ending with whitespace
✔ No entities crossing sentence boundaries

================================== Summary ==================================
✔ 6 checks passed
⚠ 1 warning


In [12]:
!python -m spacy train config.cfg --output ./output --paths.train ./train.spacy --paths.dev ./validation.spacy

✔ Created output directory: output
ℹ Saving to output directory: output
ℹ Using CPU
ℹ To switch to GPU 0, use the option: --gpu-id 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00    727.00    0.00    0.00    0.00    0.00
  0     200          0.73    765.56    0.00    0.00    0.00    0.00
  0     400          0.00      0.00    0.00    0.00    0.00    0.00
  0     600          0.00      0.00    0.00    0.00    0.00    0.00
  0     800          0.00      0.00    0.00    0.00    0.00    0.00
  0    1000          0.00      0.00    0.00    0.00    0.00    0.00
  0    1200          0.00      0.00    0.00    0.00    0.00    0.00
  0    1400    

In [13]:
!zip -r model-best.zip ./output/model-best

  adding: output/model-best/ (stored 0%)
  adding: output/model-best/meta.json (deflated 58%)
  adding: output/model-best/config.cfg (deflated 61%)
  adding: output/model-best/ner/ (stored 0%)
  adding: output/model-best/ner/model (deflated 8%)
  adding: output/model-best/ner/cfg (deflated 33%)
  adding: output/model-best/ner/moves (deflated 27%)
  adding: output/model-best/tok2vec/ (stored 0%)
  adding: output/model-best/tok2vec/model (deflated 8%)
  adding: output/model-best/tok2vec/cfg (stored 0%)
  adding: output/model-best/tokenizer (deflated 81%)
  adding: output/model-best/vocab/ (stored 0%)
  adding: output/model-best/vocab/strings.json (deflated 74%)
  adding: output/model-best/vocab/vectors (deflated 45%)
  adding: output/model-best/vocab/lookups.bin (stored 0%)
  adding: output/model-best/vocab/key2row (stored 0%)
  adding: output/model-best/vocab/vectors.cfg (stored 0%)


In [14]:
from google.colab import files
files.download("model-best.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>